<a href="https://colab.research.google.com/github/varadbarclays/dev1/blob/main/ICAIF_Chunk_Ranker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ACM ICAIF-25 AI Agentic Retrieval Grand Challenge

---

## Overview

The competition consists of two main ranking tasks:

1. **Document Ranking** – Identify and rank the five most relevant documents.  
2. **Chunk Ranking** – Identify and rank the five most relevant text chunks.

---

## Processing Pipeline

- **Data Loading**  
  Reads evaluation data from JSONL files.

- **Token Analysis**  
  Checks input size to decide whether it exceed the context length of the model..

- **Smart Ranking**  
  - *Normal cases*: Single-stage ranking.  
  - *High-token cases*: Multi-stage divide-and-conquer ranking to handle large inputs efficiently.

- **Result Compilation**  
  Combines model outputs and prepares a final CSV submission file.

---

## Output

- **kaggle_submission.csv**  
  Ready-to-submit file containing the required `sample_id` and `target_index` columns.

- **Comprehensive Statistics**  
  Summarized metrics and analysis of ranking results.

- **Top-5 Rankings**  
  Returns the five most relevant items for each query as required by the challenge.

---

## Usage

This notebook enables **end-to-end evaluation**: from loading data to generating a Kaggle-ready submission file.  
Simply run the pipeline and upload the generated `kaggle_submission.csv` to the competition platform.

In [ ]:
!pip install openai tiktoken python-dotenv pydantic tqdm

In [ ]:
import pandas as pd
import json
import ast
import re

import asyncio
import csv
import json
import os
import traceback
from typing import Dict, List

import tiktoken
from dotenv import load_dotenv
from openai import AsyncOpenAI
from pydantic import BaseModel
from tqdm.asyncio import tqdm

load_dotenv()

False

In [ ]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"varadsrivastava","key":"07be8d5799432154520a4aff77f26a0d"}'}

In [ ]:
!ls -lha kaggle.json
!pip install -q kaggle # installing the kaggle package
!mkdir -p ~/.kaggle # creating .kaggle folder where the key should be placed
!cp kaggle.json ~/.kaggle/ # move the key to the folder
!pwd # checking the present working directory

-rw-r--r-- 1 root root 71 Oct 12 20:38 kaggle.json
/content


In [ ]:
# giving rw access (if 401-nathorized)

# !chmod 600 ~/.kaggle/kaggle.json

# Dataset

In [ ]:
!kaggle competitions download -c acm-icaif-25-ai-agentic-retrieval-grand-challenge

 94% 1.16G/1.23G [00:01<00:00, 833MB/s]
100% 1.23G/1.23G [00:01<00:00, 1.00GB/s]


In [ ]:
!unzip *acm-icaif-25-ai-agentic-retrieval-grand-challenge.zip

Archive:  acm-icaif-25-ai-agentic-retrieval-grand-challenge.zip
  inflating: chunk_ranking_kaggle_dev.jsonl  
  inflating: chunk_ranking_kaggle_eval.jsonl  
  inflating: document_ranking_kaggle_dev.jsonl  
  inflating: document_ranking_kaggle_eval.jsonl  
  inflating: kaggle_submission.csv   


# Documents

In [ ]:
file_path = '/content/document_ranking_kaggle_dev.jsonl'
data = []
with open(file_path, 'r') as f:
    for line in f:
        data.append(json.loads(line))
df_doc_dev = pd.DataFrame(data)
print(f"Head of {file_path}:")
display(df_doc_dev.head())

Head of /content/document_ranking_kaggle_dev.jsonl:


,uuid,messages,qrel
0,qe100cdf8e8f5,"[{'role': 'user', 'content': 'Rank the followi...","{'4': 1, '0': 0, '1': 0, '2': 0, '3': 0}"
1,q723817294fbe,"[{'role': 'user', 'content': 'Rank the followi...","{'4': 3, '0': 2, '1': 1, '3': 1, '2': 0}"
2,qd79970afaa57,"[{'role': 'user', 'content': 'Rank the followi...","{'4': 3, '1': 2, '0': 1, '2': 1, '3': 0}"
3,qeb144d0e9991,"[{'role': 'user', 'content': 'Rank the followi...","{'1': 3, '2': 2, '4': 1, '0': 0, '3': 0}"
4,q4f1cc6ea4ba6,"[{'role': 'user', 'content': 'Rank the followi...","{'4': 3, '2': 2, '1': 1, '0': 0, '3': 0}"


In [ ]:
df_doc_dev

,uuid,messages,qrel
0,qe100cdf8e8f5,"[{'role': 'user', 'content': 'Rank the followi...","{'4': 1, '0': 0, '1': 0, '2': 0, '3': 0}"
1,q723817294fbe,"[{'role': 'user', 'content': 'Rank the followi...","{'4': 3, '0': 2, '1': 1, '3': 1, '2': 0}"
2,qd79970afaa57,"[{'role': 'user', 'content': 'Rank the followi...","{'4': 3, '1': 2, '0': 1, '2': 1, '3': 0}"
3,qeb144d0e9991,"[{'role': 'user', 'content': 'Rank the followi...","{'1': 3, '2': 2, '4': 1, '0': 0, '3': 0}"
4,q4f1cc6ea4ba6,"[{'role': 'user', 'content': 'Rank the followi...","{'4': 3, '2': 2, '1': 1, '0': 0, '3': 0}"
...,...,...,...
4981,q30f7c8056c93,"[{'role': 'user', 'content': 'Rank the followi...","{'1': 4, '2': 3, '0': 2, '4': 1, '3': 0}"
4982,q402c59dfe9b1,"[{'role': 'user', 'content': 'Rank the followi...","{'1': 4, '0': 3, '4': 2, '3': 1, '2': 0}"
4983,q45a9a878195e,"[{'role': 'user', 'content': 'Rank the followi...","{'4': 4, '2': 3, '1': 2, '0': 1, '3': 0}"
4984,q2c57b56ff808,"[{'role': 'user', 'content': 'Rank the followi...","{'4': 4, '1': 3, '2': 2, '0': 1, '3': 0}"


In [ ]:
df_doc_dev["messages"][0]

[{'role': 'user',
  'content': 'Rank the following financial document types by relevance to answer the question. Provide your ranking as a list of indices from most relevant to least relevant.\n\nQuestion: How has Agilent Technologies’ instrument reliability metric for its core diagnostics manufacturing process changed recently?\n\nDocument Types to rank:\n[Document Index 0] DEF14A\n\n[Document Index 1] 10-K\n\n[Document Index 2] 10-Q\n\n[Document Index 3] 8-K\n\n[Document Index 4] Earnings\n\nYour response must be a list of indices in exact list format (e.g., [4, 2, 1, 0, 3]), ranking every index from 0 to 4 by most relevant document type index to least relevant document type index.'}]

In [ ]:
file_path = '/content/document_ranking_kaggle_eval.jsonl'
data = []
with open(file_path, 'r') as f:
    for line in f:
        data.append(json.loads(line))
df_doc_eval = pd.DataFrame(data)
print(f"Head of {file_path}:")
display(df_doc_eval.head())

Head of /content/document_ranking_kaggle_eval.jsonl:


,_id,messages
0,doc_q39d7b7,"[{'role': 'user', 'content': 'Rank the followi..."
1,doc_q8edbb8,"[{'role': 'user', 'content': 'Rank the followi..."
2,doc_q060a50,"[{'role': 'user', 'content': 'Rank the followi..."
3,doc_q3ec868,"[{'role': 'user', 'content': 'Rank the followi..."
4,doc_qc7db20,"[{'role': 'user', 'content': 'Rank the followi..."


In [ ]:
# extract the question from df_doc_dev between "Question: " to "\n\nDocument"
df_doc_dev["question"] = df_doc_dev["messages"].apply(lambda x: x[0]["content"].split("Question: ")[1].split("\n\nDocument")[0])

df_doc_eval["question"] = df_doc_eval["messages"].apply(lambda x: x[0]["content"].split("Question: ")[1].split("\n\nDocument")[0])


In [ ]:
df_doc_dev["question"][2]

'How do sustainability or ESG considerations influence customer demand in Agilent Technologies’ market?'

In [ ]:
df_doc_eval["question"][0]

'How has Salesforce’s subscription and support segment profitability trended over recent periods?'

# Chunks

In [ ]:
file_path = '/content/chunk_ranking_kaggle_dev.jsonl'
data = []
with open(file_path, 'r') as f:
    for line in f:
        data.append(json.loads(line))
df_chunk_dev = pd.DataFrame(data)
print(f"Head of {file_path}:")
display(df_chunk_dev.head())

Head of /content/chunk_ranking_kaggle_dev.jsonl:


,uuid,messages,qrel
0,q7d7a32439929,"[{'role': 'user', 'content': 'Identify the 10 ...","{'0': 0, '1': 0, '2': 0, '3': 0, '4': 1, '5': ..."
1,qeb144d0e9991,"[{'role': 'user', 'content': 'Identify the 10 ...","{'0': 0, '1': 0, '2': 0, '3': 0, '4': 0, '5': ..."
2,qd79970afaa57,"[{'role': 'user', 'content': 'Identify the 10 ...","{'0': 0, '1': 2, '2': 0, '3': 0, '4': 0, '5': ..."
3,q4f1cc6ea4ba6,"[{'role': 'user', 'content': 'Identify the 10 ...","{'0': 0, '1': 0, '2': 0, '3': 0, '4': 0, '5': ..."
4,qc640e7d1c938,"[{'role': 'user', 'content': 'Identify the 10 ...","{'0': 0, '1': 0, '2': 0, '3': 0, '4': 0, '5': ..."


In [ ]:
df_chunk_dev

,uuid,messages,qrel
0,q7d7a32439929,"[{'role': 'user', 'content': 'Identify the 10 ...","{'0': 0, '1': 0, '2': 0, '3': 0, '4': 1, '5': ..."
1,qeb144d0e9991,"[{'role': 'user', 'content': 'Identify the 10 ...","{'0': 0, '1': 0, '2': 0, '3': 0, '4': 0, '5': ..."
2,qd79970afaa57,"[{'role': 'user', 'content': 'Identify the 10 ...","{'0': 0, '1': 2, '2': 0, '3': 0, '4': 0, '5': ..."
3,q4f1cc6ea4ba6,"[{'role': 'user', 'content': 'Identify the 10 ...","{'0': 0, '1': 0, '2': 0, '3': 0, '4': 0, '5': ..."
4,qc640e7d1c938,"[{'role': 'user', 'content': 'Identify the 10 ...","{'0': 0, '1': 0, '2': 0, '3': 0, '4': 0, '5': ..."
...,...,...,...
18850,q5dcbc1a08e24,"[{'role': 'user', 'content': 'Identify the 10 ...","{'0': 1, '1': 2, '2': 0, '3': 0, '4': 0, '5': ..."
18851,qe231f9f4a31c,"[{'role': 'user', 'content': 'Identify the 10 ...","{'0': 0, '1': 0, '2': 0, '3': 0, '4': 0, '5': ..."
18852,q402c59dfe9b1,"[{'role': 'user', 'content': 'Identify the 10 ...","{'0': 0, '1': 0, '2': 0, '3': 0, '4': 1, '5': ..."
18853,q4342a9af31f9,"[{'role': 'user', 'content': 'Identify the 10 ...","{'0': 0, '1': 0, '2': 0, '3': 0, '4': 0, '5': ..."


In [ ]:
df_chunk_dev["qrel"][0]

{'0': 0,
 '1': 0,
 '2': 0,
 '3': 0,
 '4': 1,
 '5': 0,
 '6': 0,
 '7': 0,
 '8': 2,
 '9': 0,
 '10': 0,
 '11': 0,
 '12': 0,
 '13': 0,
 '14': 0,
 '15': 0,
 '16': 0,
 '17': 0,
 '18': 0,
 '19': 0,
 '20': 0,
 '21': 0,
 '22': 0,
 '23': 0,
 '24': 2,
 '25': 0,
 '26': 0,
 '27': 0,
 '28': 0,
 '29': 0,
 '30': 0,
 '31': 0,
 '32': 0,
 '33': 0,
 '34': 0,
 '35': 0,
 '36': 0,
 '37': 0,
 '38': 0,
 '39': 0,
 '40': 0,
 '41': 0,
 '42': 1,
 '43': 0,
 '44': 0,
 '45': 0,
 '46': 0,
 '47': 0,
 '48': 0,
 '49': 0,
 '50': 0,
 '51': 0,
 '52': 0,
 '53': 0,
 '54': 0,
 '55': 0,
 '56': 0,
 '57': 0,
 '58': 0,
 '59': 0,
 '60': 0,
 '61': 0,
 '62': 0,
 '63': 0,
 '64': 2,
 '65': 0,
 '66': 0,
 '67': 0,
 '68': 0,
 '69': 0,
 '70': 0,
 '71': 0,
 '72': 0,
 '73': 0,
 '74': 0,
 '75': 0,
 '76': 0,
 '77': 0,
 '78': 0,
 '79': 0,
 '80': 0,
 '81': 0,
 '82': 0,
 '83': 0,
 '84': 0,
 '85': 0,
 '86': 0,
 '87': 0,
 '88': 0,
 '89': 0,
 '90': 0,
 '91': 0,
 '92': 0,
 '93': 0,
 '94': 0,
 '95': 0,
 '96': 0,
 '97': 0,
 '98': 0,
 '99': 0,
 '100': 0,

In [ ]:
file_path = '/content/chunk_ranking_kaggle_eval.jsonl'
data = []
with open(file_path, 'r') as f:
    for line in f:
        data.append(json.loads(line))
df_chunk_eval = pd.DataFrame(data)
print(f"Head of {file_path}:")
display(df_chunk_eval.head())

Head of /content/chunk_ranking_kaggle_eval.jsonl:


,_id,messages
0,chunk_q613509,"[{'role': 'user', 'content': 'Identify the 10 ..."
1,chunk_q83d560,"[{'role': 'user', 'content': 'Identify the 10 ..."
2,chunk_qc06926,"[{'role': 'user', 'content': 'Identify the 10 ..."
3,chunk_qaae0f2,"[{'role': 'user', 'content': 'Identify the 10 ..."
4,chunk_q059068,"[{'role': 'user', 'content': 'Identify the 10 ..."


In [ ]:
# extract the chunks from df_chunk_dev between "(best first).\n" to "\n- Put the BEST chunk"
df_chunk_dev["chunks"] = df_chunk_dev["messages"].apply(lambda x: x[0]["content"].split("(best first).\n")[1].split("\n\nTask: Select and rank")[0])

df_chunk_eval["chunks"] = df_chunk_eval["messages"].apply(lambda x: x[0]["content"].split("(best first).\n")[1].split("\n\nTask: Select and rank")[0])


In [ ]:
len(df_chunk_dev["messages"][0][0]["content"])

298154

In [ ]:
len(df_chunk_dev["chunks"][0])

297719

In [ ]:
df_chunk_dev["chunks"][0]

'Question: What share ownership guidelines are defined for Agilent Technologies’ executives and directors?\nText chunks:\n[Chunk Index 0] # UNITED STATES\nSECURITIES AND EXCHANGE COMMISSION\nWashington, D.C. 20549\n\n# SCHEDULE 14A INFORMATION\n\n# PROXY STATEMENT PURSUANT TO SECTION 14(a) OF THE\nSECURITIES EXCHANGE ACT OF 1934\n(AMENDMENT NO. )\nSCHEDULE 14A\n\nFiled by:\n[x] Filed by the Registrant\n[ ] Filed by a Party other than the Registrant\n\nCheck the appropriate box:\n[ ] Preliminary Proxy Statement\n[ ] Confidential, for Use of the Commission Only (as permitted by Rule 14a-6(e)(2))\n[x] Definitive Proxy Statement\n[ ] Definitive Additional Materials\n[ ] Soliciting Material Pursuant to §240.14a-12\n\n## AGILENT TECHNOLOGIES, INC.\n\n(Name of Registrant as Specified In Its Charter)\n\n(Name of Person(s) Filing Proxy Statement, if other than the Registrant)\n\nPayment of Filing Fee (Check the appropriate box):\n\n- No fee required.\n\n- Fee paid previously with preliminary ma

In [ ]:
df_chunk_dev["messages"][0]

[{'role': 'user',
  'content': 'Identify the 10 most relevant text chunks for answering this question, then rank them in order of relevance (best first).\nQuestion: What share ownership guidelines are defined for Agilent Technologies’ executives and directors?\nText chunks:\n[Chunk Index 0] # UNITED STATES\nSECURITIES AND EXCHANGE COMMISSION\nWashington, D.C. 20549\n\n# SCHEDULE 14A INFORMATION\n\n# PROXY STATEMENT PURSUANT TO SECTION 14(a) OF THE\nSECURITIES EXCHANGE ACT OF 1934\n(AMENDMENT NO. )\nSCHEDULE 14A\n\nFiled by:\n[x] Filed by the Registrant\n[ ] Filed by a Party other than the Registrant\n\nCheck the appropriate box:\n[ ] Preliminary Proxy Statement\n[ ] Confidential, for Use of the Commission Only (as permitted by Rule 14a-6(e)(2))\n[x] Definitive Proxy Statement\n[ ] Definitive Additional Materials\n[ ] Soliciting Material Pursuant to §240.14a-12\n\n## AGILENT TECHNOLOGIES, INC.\n\n(Name of Registrant as Specified In Its Charter)\n\n(Name of Person(s) Filing Proxy Stateme

In [ ]:
# df_chunk_eval["chunks"][5]

In [ ]:
df_chunk_dev

,uuid,messages,qrel,chunks
0,q7d7a32439929,"[{'role': 'user', 'content': 'Identify the 10 ...","{'0': 0, '1': 0, '2': 0, '3': 0, '4': 1, '5': ...",Question: What share ownership guidelines are ...
1,qeb144d0e9991,"[{'role': 'user', 'content': 'Identify the 10 ...","{'0': 0, '1': 0, '2': 0, '3': 0, '4': 0, '5': ...",Question: What exposure does Agilent Technolog...
2,qd79970afaa57,"[{'role': 'user', 'content': 'Identify the 10 ...","{'0': 0, '1': 2, '2': 0, '3': 0, '4': 0, '5': ...",Question: How do sustainability or ESG conside...
3,q4f1cc6ea4ba6,"[{'role': 'user', 'content': 'Identify the 10 ...","{'0': 0, '1': 0, '2': 0, '3': 0, '4': 0, '5': ...",Question: What sentiment trends are visible ar...
4,qc640e7d1c938,"[{'role': 'user', 'content': 'Identify the 10 ...","{'0': 0, '1': 0, '2': 0, '3': 0, '4': 0, '5': ...",Question: What did Agilent Technologies’ execu...
...,...,...,...,...
18850,q5dcbc1a08e24,"[{'role': 'user', 'content': 'Identify the 10 ...","{'0': 1, '1': 2, '2': 0, '3': 0, '4': 0, '5': ...",Question: What was the level of Zoetis Inc.’s ...
18851,qe231f9f4a31c,"[{'role': 'user', 'content': 'Identify the 10 ...","{'0': 0, '1': 0, '2': 0, '3': 0, '4': 0, '5': ...",Question: How are talent retention or workforc...
18852,q402c59dfe9b1,"[{'role': 'user', 'content': 'Identify the 10 ...","{'0': 0, '1': 0, '2': 0, '3': 0, '4': 1, '5': ...",Question: What themes have investors highlight...
18853,q4342a9af31f9,"[{'role': 'user', 'content': 'Identify the 10 ...","{'0': 0, '1': 0, '2': 0, '3': 0, '4': 0, '5': ...",Question: What guidance was offered regarding ...


In [ ]:
print(len(df_chunk_eval), len(df_chunk_eval["_id"].unique()))

200 200


In [ ]:
# Create a counter for each unique UUID
uuid_counts = {}
new_uuids = []

for uuid in df_chunk_dev['uuid']:
    if uuid not in uuid_counts:
        uuid_counts[uuid] = 0
        new_uuids.append(f"{uuid}_doc{uuid_counts[uuid]}")
    else:
        uuid_counts[uuid] += 1
        new_uuids.append(f"{uuid}_doc{uuid_counts[uuid]}")

df_chunk_dev['uuid'] = new_uuids

print("Unique UUIDs created for df_chunk_dev.")
display(df_chunk_dev.head())

Unique UUIDs created for df_chunk_dev.


,uuid,messages,qrel,chunks
0,q7d7a32439929_doc0,"[{'role': 'user', 'content': 'Identify the 10 ...","{'0': 0, '1': 0, '2': 0, '3': 0, '4': 1, '5': ...",Question: What share ownership guidelines are ...
1,qeb144d0e9991_doc0,"[{'role': 'user', 'content': 'Identify the 10 ...","{'0': 0, '1': 0, '2': 0, '3': 0, '4': 0, '5': ...",Question: What exposure does Agilent Technolog...
2,qd79970afaa57_doc0,"[{'role': 'user', 'content': 'Identify the 10 ...","{'0': 0, '1': 2, '2': 0, '3': 0, '4': 0, '5': ...",Question: How do sustainability or ESG conside...
3,q4f1cc6ea4ba6_doc0,"[{'role': 'user', 'content': 'Identify the 10 ...","{'0': 0, '1': 0, '2': 0, '3': 0, '4': 0, '5': ...",Question: What sentiment trends are visible ar...
4,qc640e7d1c938_doc0,"[{'role': 'user', 'content': 'Identify the 10 ...","{'0': 0, '1': 0, '2': 0, '3': 0, '4': 0, '5': ...",Question: What did Agilent Technologies’ execu...


In [ ]:
len(df_chunk_dev["uuid"].unique())

18855

## For chunks:
For annotating those relevance scores, we followed
TREC Eval: 0 (irrelevant), 1 (partially relevant), and 2 (directly relevant).

## Step 1: Differentiate between 2/1 and 0

In [ ]:
# Expected data layout (JSONL)

# train.jsonl: one object per (query, chunk, label)

# {"query_id":"Q1","query":"What are Apple’s 2023 gross margins?","chunk_id":"C1","chunk_text":"Item 7. Management’s Discussion... gross margin was 43% in 2023 ...","label":2}
# {"query_id":"Q1","query":"What are Apple’s 2023 gross margins?","chunk_id":"C2","chunk_text":"Risk Factors...","label":0}
# {"query_id":"Q2","query":"Did the company raise FY24 guidance?","chunk_id":"C9","chunk_text":"Earnings Call Q&A ... raised revenue guidance for FY24 ...","label":1}


# dev_corpus.jsonl: passage corpus (chunk_id → text)

# dev_queries.jsonl: eval queries (query_id → text)

# dev_qrels.jsonl: graded relevance (query_id, chunk_id, label ∈ {0,1,2})

# You can export these from your loader; labels 2/1 count as relevant for retrieval metrics.


In [ ]:
# add a

In [ ]:
from concurrent.futures import ProcessPoolExecutor, as_completed
import re

chunk_pattern = re.compile(
    r'\[Chunk Index (\d+)\]\s*([\s\S]*?)(?=\[Chunk Index|\bTask:|$)',
    # flags=re.MULTILINE
)

def process_row(args):
    idx, messages, uuid, qrel = args
    out = []
    try:
        content = messages[0]["content"]
    except Exception:
        return out

    question = None
    if "Question:" in content:
        part = content.split("Question:", 1)[1]
        question = part.splitlines()[0].strip() if part else None

    for m in chunk_pattern.finditer(content):
        orig_idx = int(m.group(1))
        chunk_text = m.group(2).strip()
        if "Task:" in chunk_text:
            chunk_text = chunk_text.split("Task:", 1)[0].strip()
        if not chunk_text:
            continue

        label = None
        try:
            label = qrel.get(str(orig_idx))
        except Exception:
            try:
                label = qrel.get(orig_idx)
            except Exception:
                label = None

        out.append({
            "query_id": uuid,
            "query": question,
            "chunk_id": uuid + "_chunk" + str(orig_idx),
            "chunk_text": chunk_text,
            "label": label
        })
    return out

# Build list of args
args_list = [(i, row.messages, row.uuid, row.qrel) for i, row in enumerate(df_chunk_dev.itertuples(index=False))]

results = []
with ProcessPoolExecutor() as ex:
    futures = {ex.submit(process_row, a): a for a in args_list}
    for fut in tqdm(as_completed(futures), total=len(futures)):
        results.extend(fut.result())

df_chunk_dev_sep_chunks = pd.DataFrame(results)


100%|██████████| 18855/18855 [01:41<00:00, 184.98it/s]


In [ ]:
df_chunk_dev_sep_chunks["query_id"][0]

'q7d7a32439929_doc0'

In [ ]:
df_chunk_dev_sep_chunks

,query_id,query,chunk_id,chunk_text,label
0,q7d7a32439929_doc0,What share ownership guidelines are defined fo...,q7d7a32439929_doc0_chunk0,# UNITED STATES\nSECURITIES AND EXCHANGE COMMI...,0
1,q7d7a32439929_doc0,What share ownership guidelines are defined fo...,q7d7a32439929_doc0_chunk1,"# 5301 Stevens Creek Boulevard Santa Clara, Ca...",0
2,q7d7a32439929_doc0,What share ownership guidelines are defined fo...,q7d7a32439929_doc0_chunk2,"# 5301 Stevens Creek Boulevard Santa Clara, Ca...",0
3,q7d7a32439929_doc0,What share ownership guidelines are defined fo...,q7d7a32439929_doc0_chunk3,# CAUTIONARY NOTE REGARDING FORWARD-LOOKING ST...,0
4,q7d7a32439929_doc0,What share ownership guidelines are defined fo...,q7d7a32439929_doc0_chunk4,# Director Nominees (cont.)\n\n## Corporate Go...,1
...,...,...,...,...,...
3193196,q4342a9af31f9_doc3,What guidance was offered regarding Zoetis Inc...,q4342a9af31f9_doc3_chunk262,SIGNATURES\n\nPursuant to the requirements of ...,0
3193197,q4342a9af31f9_doc3,What guidance was offered regarding Zoetis Inc...,q4342a9af31f9_doc3_chunk263,"####Zoetis Inc.##\nDated: February 13, 2024##B...",0
3193198,q4342a9af31f9_doc3,What guidance was offered regarding Zoetis Inc...,q4342a9af31f9_doc3_chunk264,"We, the undersigned directors and officers of ...",0
3193199,q4342a9af31f9_doc3,What guidance was offered regarding Zoetis Inc...,q4342a9af31f9_doc3_chunk265,Name####Title##Date\n/S/ KRISTIN C. PECK####Ch...,0


In [ ]:
df_chunk_dev.head()

,uuid,messages,qrel,chunks
0,q7d7a32439929_doc0,"[{'role': 'user', 'content': 'Identify the 10 ...","{'0': 0, '1': 0, '2': 0, '3': 0, '4': 1, '5': ...",Question: What share ownership guidelines are ...
1,qeb144d0e9991_doc0,"[{'role': 'user', 'content': 'Identify the 10 ...","{'0': 0, '1': 0, '2': 0, '3': 0, '4': 0, '5': ...",Question: What exposure does Agilent Technolog...
2,qd79970afaa57_doc0,"[{'role': 'user', 'content': 'Identify the 10 ...","{'0': 0, '1': 2, '2': 0, '3': 0, '4': 0, '5': ...",Question: How do sustainability or ESG conside...
3,q4f1cc6ea4ba6_doc0,"[{'role': 'user', 'content': 'Identify the 10 ...","{'0': 0, '1': 0, '2': 0, '3': 0, '4': 0, '5': ...",Question: What sentiment trends are visible ar...
4,qc640e7d1c938_doc0,"[{'role': 'user', 'content': 'Identify the 10 ...","{'0': 0, '1': 0, '2': 0, '3': 0, '4': 0, '5': ...",Question: What did Agilent Technologies’ execu...


In [ ]:
df_doc_dev.head()

,uuid,messages,qrel,question
0,qe100cdf8e8f5,"[{'role': 'user', 'content': 'Rank the followi...","{'4': 1, '0': 0, '1': 0, '2': 0, '3': 0}",How has Agilent Technologies’ instrument relia...
1,q723817294fbe,"[{'role': 'user', 'content': 'Rank the followi...","{'4': 3, '0': 2, '1': 1, '3': 1, '2': 0}",How did analysts question the outlook for Agil...
2,qd79970afaa57,"[{'role': 'user', 'content': 'Rank the followi...","{'4': 3, '1': 2, '0': 1, '2': 1, '3': 0}",How do sustainability or ESG considerations in...
3,qeb144d0e9991,"[{'role': 'user', 'content': 'Rank the followi...","{'1': 3, '2': 2, '4': 1, '0': 0, '3': 0}","What exposure does Agilent Technologies, Inc. ..."
4,q4f1cc6ea4ba6,"[{'role': 'user', 'content': 'Rank the followi...","{'4': 3, '2': 2, '1': 1, '0': 0, '3': 0}",What sentiment trends are visible around Agile...


In [ ]:
df_doc_dev[df_doc_dev["uuid"] == "q4342a9af31f9"]

,uuid,messages,qrel,question
4980,q4342a9af31f9,"[{'role': 'user', 'content': 'Rank the followi...","{'4': 4, '3': 3, '1': 2, '2': 1, '0': 0}",What guidance was offered regarding Zoetis Inc...


In [ ]:
len(df_doc_dev)

4986

In [ ]:
len(df_chunk_dev)

18855

In [ ]:
len(df_chunk_dev["uuid"].unique())

18855

In [ ]:
df_chunk_dev_sep_chunks

,query_id,query,chunk_id,chunk_text,label
0,q7d7a32439929_doc0,What share ownership guidelines are defined fo...,q7d7a32439929_doc0_chunk0,# UNITED STATES\nSECURITIES AND EXCHANGE COMMI...,0
1,q7d7a32439929_doc0,What share ownership guidelines are defined fo...,q7d7a32439929_doc0_chunk1,"# 5301 Stevens Creek Boulevard Santa Clara, Ca...",0
2,q7d7a32439929_doc0,What share ownership guidelines are defined fo...,q7d7a32439929_doc0_chunk2,"# 5301 Stevens Creek Boulevard Santa Clara, Ca...",0
3,q7d7a32439929_doc0,What share ownership guidelines are defined fo...,q7d7a32439929_doc0_chunk3,# CAUTIONARY NOTE REGARDING FORWARD-LOOKING ST...,0
4,q7d7a32439929_doc0,What share ownership guidelines are defined fo...,q7d7a32439929_doc0_chunk4,# Director Nominees (cont.)\n\n## Corporate Go...,1
...,...,...,...,...,...
3193196,q4342a9af31f9_doc3,What guidance was offered regarding Zoetis Inc...,q4342a9af31f9_doc3_chunk262,SIGNATURES\n\nPursuant to the requirements of ...,0
3193197,q4342a9af31f9_doc3,What guidance was offered regarding Zoetis Inc...,q4342a9af31f9_doc3_chunk263,"####Zoetis Inc.##\nDated: February 13, 2024##B...",0
3193198,q4342a9af31f9_doc3,What guidance was offered regarding Zoetis Inc...,q4342a9af31f9_doc3_chunk264,"We, the undersigned directors and officers of ...",0
3193199,q4342a9af31f9_doc3,What guidance was offered regarding Zoetis Inc...,q4342a9af31f9_doc3_chunk265,Name####Title##Date\n/S/ KRISTIN C. PECK####Ch...,0


In [ ]:
df_chunk_dev_sep_chunks.to_csv("df_chunk_dev_sep_chunks.csv", index=False)

In [ ]:
len(df_chunk_dev_sep_chunks)

3193201

In [ ]:
import argparse, json, random, math, os
from collections import defaultdict
from typing import Dict, List, Tuple

from torch.utils.data import Dataset, DataLoader
from sentence_transformers import SentenceTransformer, InputExample, losses, util, evaluation, models
from sentence_transformers.evaluation import InformationRetrievalEvaluator
import torch
from tqdm import tqdm

In [ ]:
df_chunk_dev_sep_chunks = pd.read_csv("df_chunk_dev_sep_chunks.csv")

In [ ]:
# utility

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                yield json.loads(line)

def decide_prefixes(base_model: str):
    """
    Returns (q_prefix, p_prefix) recommended for the model family.
    BGE/E5 both benefit from 'query: ' / 'passage: ' style prefixes.
    """
    low = base_model.lower()
    if "bge" in low:
        return "query: ", "passage: "
    if "e5" in low:
        # E5 style also commonly uses 'query: ' / 'passage: '
        return "query: ", "passage: "
    # Fallback
    return "", ""

In [ ]:
# ----------------------------
# Dataset: one positive per (query_id), in-batch negatives between queries
# Optionally include a second view with a hard negative for multi-view InfoNCE
# ----------------------------

class FinAgentPairs(Dataset):
    def __init__(self, train_file, q_prefix: str, p_prefix: str, include_hard_neg: bool = True):
        """
        Builds (query, pos) pairs; keeps per-query 0-labeled chunks to sample as hard negatives.
        """
        self.q_prefix = q_prefix
        self.p_prefix = p_prefix
        self.include_hard_neg = include_hard_neg

        by_query_pos: Dict[str, List[Tuple[str, str]]] = defaultdict(list)   # qid -> list of (chunk_id, text) where label>=1
        by_query_neg: Dict[str, List[Tuple[str, str]]] = defaultdict(list)   # qid -> list of (chunk_id, text) where label==0
        self.query_text: Dict[str, str] = {}

        for i in range(len(df_chunk_dev_sep_chunks)):
            obj = df_chunk_dev_sep_chunks.iloc[i]
            qid = obj["query_id"]
            qtxt = obj["query"]
            cid = obj["chunk_id"]
            ctxt = obj["chunk_text"]
            label = int(obj["label"])
            self.query_text[qid] = qtxt
            if label >= 1:
                by_query_pos[qid].append((cid, ctxt))
            else:
                by_query_neg[qid].append((cid, ctxt))

        # Build one positive per (qid, sampled pos chunk) instance.
        self.samples = []
        for qid, pos_list in by_query_pos.items():
            if not pos_list:  # skip rare empty
                continue
            for cid, ctxt in pos_list:
                hard_negs = by_query_neg.get(qid, [])
                self.samples.append({
                    "query_id": qid,
                    "query": self.q_prefix + self.query_text[qid],
                    "pos_text": self.p_prefix + ctxt,
                    "hard_negs": [self.p_prefix + nctxt for _, nctxt in hard_negs] if hard_negs else []
                })

        random.shuffle(self.samples)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        # Primary pair for MultipleNegativesRankingLoss:
        ex = InputExample(texts=[s["query"], s["pos_text"]])

        # Optionally add a multi-view form with a sampled hard negative.
        # Many folks simply rely on in-batch negatives; this adds a second view to strengthen within-query discrimination.
        if self.include_hard_neg and s["hard_negs"]:
            neg_text = random.choice(s["hard_negs"])
            # Tri-flow (anchor, pos, neg) for InfoNCE-like losses; MNRL ignores 3rd, but some variants accept it.
            # We'll return it in 'metadata' so a custom collate could use it if you switch to a triplet loss later.
            ex.metadata = {"hard_neg": neg_text}
        return ex

In [ ]:
import os, json, random, pickle
from collections import defaultdict
from typing import List, Dict, Any
from sentence_transformers import InputExample

class StreamingFinAgentPairs:
    """
    Memory-efficient dataset: store file offsets (ints) and query texts in memory,
    load chunk lines on demand by seeking into the jsonl file.
    """
    def __init__(self,
                 train_file: str,
                 q_prefix: str,
                 p_prefix: str,
                 include_hard_neg: bool = True,
                 index_path: str = None,
                 rebuild_index: bool = False,
                 seed: int = 42):
        """
        train_file: path to jsonl lines {'query_id','query','chunk_id','chunk_text','label'}
        index_path: optional path to save/load prebuilt index (.pkl)
        rebuild_index: force rebuild the index even if index_path exists
        """
        random.seed(seed)
        self.train_file = train_file
        self.q_prefix = q_prefix
        self.p_prefix = p_prefix
        self.include_hard_neg = include_hard_neg

        # Index structure we will keep in memory (compact)
        # queries[qid] = query_text
        # pos_index[qid] = list of offsets where label >=1 lines are located (file offsets)
        # neg_index[qid] = list of offsets where label == 0 lines are located
        self.queries: Dict[str, str] = {}
        self.pos_index: Dict[str, List[int]] = defaultdict(list)
        self.neg_index: Dict[str, List[int]] = defaultdict(list)
        self._ordered_pos_items = []  # list of tuples (qid, offset) to emulate one-sample-per-pos behavior

        # Try to load index if given
        if index_path and os.path.exists(index_path) and not rebuild_index:
            print(f"Loading index from {index_path} ...")

            # idx = pd.read_csv(index_path)
            with open(index_path, "rb") as fh:
                idx = pickle.load(fh)
            self.queries = idx["queries"]
            self.pos_index = idx["pos_index"]
            self.neg_index = idx["neg_index"]
            self._ordered_pos_items = idx.get("ordered_pos_items", [])
            print("Index loaded. Pos queries:", len(self.pos_index), "Neg queries:", len(self.neg_index))
        else:
            print("Building index from train file (one pass)...")
            self._build_index_and_ordered_list()
            if index_path:
                print(f"Saving index to {index_path} ...")
                # temp = pd.DataFrame({
                #     "queries": self.queries,
                #     "pos_index": self.pos_index,
                #     "neg_index": self.neg_index,
                #     "ordered_pos_items": self._ordered_pos_items
                # })
                # temp.to_csv(index_path)

                with open(index_path, "wb") as fh:
                    pickle.dump({
                        "queries": self.queries,
                        "pos_index": self.pos_index,
                        "neg_index": self.neg_index,
                        "ordered_pos_items": self._ordered_pos_items
                    }, fh)

    def _build_index_and_ordered_list(self):
        """
        Walk the jsonl file, keep offsets for each line and map offsets to pos/neg lists.
        """
        for i in range(len(df_chunk_dev_sep_chunks)):
          obj = df_chunk_dev_sep_chunks.iloc[i]
          qid = obj["query_id"]
          qtxt = obj["query"]
          cid = obj["chunk_id"]
          ctxt = obj["chunk_text"]
          label = int(obj["label"])
          # open in binary to get reliable tell() offsets
            # while True:
            #     offset = f.tell()
            #     line = f.readline()
            #     if not line:
            #         break
            #     try:
            #         obj = json.loads(line.decode("utf-8"))
            #     except Exception as e:
            #         # skip malformed line
            #         continue
          # qid = obj.get("query_id")
          # qtxt = obj.get("query", "")
          # cid = obj.get("chunk_id")
          # label = int(obj.get("label", 0))
          # store query text (last occurrence will match; queries are small)
          if qid not in self.queries:
              self.queries[qid] = qtxt
          # store offsets
          if label >= 1:
              self.pos_index[qid].append(i)
              # track ordered pos items for constructing dataset length & one-sample-per-pos behavior
              self._ordered_pos_items.append((qid, i))
          else:
              self.neg_index[qid].append(i)

        random.shuffle(self._ordered_pos_items)
        print("Indexed train file: queries =", len(self.queries),
              "total pos samples =", len(self._ordered_pos_items))

    def __len__(self):
        # we create one training sample per positive chunk occurrence (like before).
        return len(self._ordered_pos_items)

    def _read_line_at_offset(self, offset: int) -> dict:
        # read single json line at offset
        return df_chunk_dev_sep_chunks.iloc[offset]

    def __getitem__(self, idx: int) -> InputExample:
        """
        Returns InputExample(texts=[query_with_prefix, pos_with_prefix])
        and stores metadata.hard_neg if include_hard_neg is True and available.
        """
        qid, pos_offset = self._ordered_pos_items[idx]
        pos_obj = self._read_line_at_offset(pos_offset)
        pos_text = pos_obj["chunk_text"]

        q_text = self.queries[qid]

        # build example
        ex = InputExample(texts=[self.q_prefix + q_text, self.p_prefix + pos_text])

        # optionally sample a hard negative from same query (label 0)
        if self.include_hard_neg and self.neg_index.get(qid):
            neg_offset = random.choice(self.neg_index[qid])
            neg_obj = self._read_line_at_offset(neg_offset)
            ex.metadata = {"hard_neg": self.p_prefix + neg_obj["chunk_text"]}
        return ex


In [ ]:
# ----------------------------
# Dev evaluator (IR)
# ----------------------------

def build_ir_evaluator(dev_corpus_file, q_prefix, p_prefix):
    corpus = {}     # pid -> text
    queries = {}    # qid -> text
    relevant_docs = defaultdict(dict)  # qid -> {pid: relevance}

    for i in range(len(dev_corpus_file)):
      obj = dev_corpus_file.iloc[i]
      pid = obj["chunk_id"]
      corpus[pid] = p_prefix + obj["chunk_text"]

      qid = obj["query_id"]
      queries[qid] = q_prefix + obj["query"]

      label = int(obj["label"])
      if label >= 1:  # treat label≥1 as relevant for retrieval metrics; graded metrics computed internally
        relevant_docs[qid][pid] = label

    # for obj in load_jsonl(dev_corpus_file):
    #     pid = obj["chunk_id"]
    #     corpus[pid] = p_prefix + obj["chunk_text"]

    # for obj in load_jsonl(dev_queries_file):
    #     qid = obj["query_id"]
    #     queries[qid] = q_prefix + obj["query"]

    # for obj in load_jsonl(dev_qrels_file):
    #     qid = obj["query_id"]; pid = obj["chunk_id"]; label = int(obj["label"])
    #     if label >= 1:  # treat label≥1 as relevant for retrieval metrics; graded metrics computed internally
    #         relevant_docs[qid][pid] = label

    return InformationRetrievalEvaluator(
        queries=queries,
        corpus=corpus,
        relevant_docs=relevant_docs,
        name="finagentbench-dev",
        score_functions={"cos_sim": util.cos_sim},
        show_progress_bar=True
    )

In [ ]:
df_chunk_dev_sep_chunks.head()

,query_id,query,chunk_id,chunk_text,label
0,q7d7a32439929_doc0,What share ownership guidelines are defined fo...,q7d7a32439929_doc0_chunk0,# UNITED STATES\nSECURITIES AND EXCHANGE COMMI...,0
1,q7d7a32439929_doc0,What share ownership guidelines are defined fo...,q7d7a32439929_doc0_chunk1,"# 5301 Stevens Creek Boulevard Santa Clara, Ca...",0
2,q7d7a32439929_doc0,What share ownership guidelines are defined fo...,q7d7a32439929_doc0_chunk2,"# 5301 Stevens Creek Boulevard Santa Clara, Ca...",0
3,q7d7a32439929_doc0,What share ownership guidelines are defined fo...,q7d7a32439929_doc0_chunk3,# CAUTIONARY NOTE REGARDING FORWARD-LOOKING ST...,0
4,q7d7a32439929_doc0,What share ownership guidelines are defined fo...,q7d7a32439929_doc0_chunk4,# Director Nominees (cont.)\n\n## Corporate Go...,1


In [ ]:
train_file = df_chunk_dev_sep_chunks
dev_corpus = df_chunk_dev_sep_chunks
# dev_corpus = df_chunk_dev_sep_chunks["chunk_text"]
# dev_queries = df_chunk_dev_sep_chunks["query_id"]
# dev_qrels = df_chunk_dev_sep_chunks["qrel"]
base_model = "BAAI/bge-large-en-v1.5"
output_dir = "output"
batch_size = 64
epochs = 2
lr = 2e-5
max_seq_length = 512
warmup_ratio = 0.1
fp16 = True
seed = 42
include_hard_neg = "True"

random.seed(seed)
torch.manual_seed(seed)

os.makedirs(output_dir, exist_ok=True)

In [ ]:
q_prefix, p_prefix = decide_prefixes(base_model)

print(f"Loading base model: {base_model}")
model = SentenceTransformer(base_model)
model.max_seq_length = max_seq_length

Loading base model: BAAI/bge-large-en-v1.5


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [ ]:
# Training data

# train_ds = FinAgentPairs(train_file, q_prefix, p_prefix, include_hard_neg=include_hard_neg)

# IMPORTANT: ensure one pair per unique query per batch to maximize in-batch negatives.
# We do that by sorting by query_id and using a sampler that picks distinct queries.
# Simpler practical trick: shuffle samples globally and set batch_size so that probability of duplicate qids within a batch is low.
# For strictness you can implement a custom sampler; for most FinAgentBench sizes, BS=64 with shuffle=True works well.

# train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=True)

In [ ]:
train_ds = StreamingFinAgentPairs(
    train_file=train_file,
    q_prefix=q_prefix,
    p_prefix=p_prefix,
    include_hard_neg=True,
    index_path="train_index.pkl",   # optional cache
    rebuild_index=False
)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=4, pin_memory=False)

Building index from train file (one pass)...
Indexed train file: queries = 18855 total pos samples = 164066
Saving index to train_index.pkl ...


In [ ]:
# Loss: MultipleNegativesRankingLoss (InfoNCE for bi-encoders)
train_loss = losses.MultipleNegativesRankingLoss(model)

In [ ]:
# Dev evaluator
evaluator = build_ir_evaluator(dev_corpus, q_prefix, p_prefix)

warmup_steps = math.ceil(len(train_loader) * epochs * warmup_ratio)

In [ ]:
# Train
os.environ["WANDB_DISABLED"] = "true"

model.fit(
    train_objectives=[(train_loader, train_loss)],
    epochs=epochs,
    optimizer_params={"lr": lr},
    warmup_steps=warmup_steps,
    output_path=output_dir,
    save_best_model=True,
    evaluator=evaluator,
    evaluation_steps=max(100, len(train_loader)//5),
    use_amp=fp16,
    show_progress_bar=True,
)

print("Finished training. Saving final model...")
model.save(output_dir)

# Final evaluation
print("Running final evaluation on dev …")
model = SentenceTransformer(output_dir)
evaluator(model, output_path=output_dir)

print("Done. Model at:", output_dir)

## DeepSeek

In [ ]:
client = AsyncOpenAI(api_key=DEEPSEEK_API_KEY, base_url="https://api.deepseek.com")

In [ ]:
async def deepseek_model_chunks(system_prompt, user_prompt):
    """Generates Question-Thought-Answer triplets from the given text using OpenAI."""
    try:
        response = await client.chat.completions.create(
            model="deepseek-chat",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            # response_format={"type": "json_object"}
        )

        # print(response)

        # Attempt to parse the response as JSON
        # try:
        #     rankings = json.loads(response.choices[0].message.content)
        #     return rankings
        # except json.JSONDecodeError as e:
        #     print(f"Error decoding JSON from model response: {e}")
        #     print("Raw response text:", response.choices[0].message.content)
        #     return []

        return response

    except Exception as e:
        print(f"An error occurred during ranking generation: {e}")
        return []

In [ ]:
deepseek_prompts = []

system_prompt_chunk = """You are a helpful assistant that provides financial document chunk rankings based on which chunk is most likely to contain the information relevant to the question. Do not add any thinking, reasoning or other texts."""

for index, row in df_chunk_eval.iterrows():
    eval_question_chunk = row['chunks']

    prompt = f"""Evaluate the relevance ranking of the chunks for the following question based on how relevant the content in chunk is for answering the question.\n\n{eval_question_chunk}\n\n"""

    prompt += """Your Task:\nRank the chunks on one of the three numbers: 0 (chunk is irrelevant in answering the question asked), 1 (chunk is partially relevant in answering the question asked), and 2 (chunk is directly relevant in answering the question asked). Return the result rankings in a JSON format in the form: {'0':1, '1':0..} where keys are the chunk numbers and values are the ranks assigned.\nAnswer Ranking:"""

    deepseek_prompts.append(prompt)

df_chunk_eval['deepseek_prompts'] = deepseek_prompts

print("DeepSeek prompts generated and added to df_doc_eval.")
display(df_chunk_eval.head())

DeepSeek prompts generated and added to df_doc_eval.


,_id,messages,chunks,deepseek_prompts
0,chunk_q613509,"[{'role': 'user', 'content': 'Identify the 10 ...",Question: What guidance was offered on Ball Co...,Evaluate the relevance ranking of the chunks f...
1,chunk_q83d560,"[{'role': 'user', 'content': 'Identify the 10 ...",Question: How has the ratio of BlackRock’s rec...,Evaluate the relevance ranking of the chunks f...
2,chunk_qc06926,"[{'role': 'user', 'content': 'Identify the 10 ...",Question: How has Caterpillar Inc.’s construct...,Evaluate the relevance ranking of the chunks f...
3,chunk_qaae0f2,"[{'role': 'user', 'content': 'Identify the 10 ...",Question: How do global economic or geopolitic...,Evaluate the relevance ranking of the chunks f...
4,chunk_q059068,"[{'role': 'user', 'content': 'Identify the 10 ...",Question: What did Chubb Limited’s leadership ...,Evaluate the relevance ranking of the chunks f...


In [ ]:
df_chunk_eval["deepseek_prompts"][0]

"Evaluate the relevance ranking of the chunks for the following question based on how relevant the content in chunk is for answering the question.\n\nQuestion: What guidance was offered on Ball Corporation’s aluminum can production capacity targets?\nText chunks:\n[Chunk Index 0] PART I. FINANCIAL INFORMATION\n\nItem 1. FINANCIAL STATEMENTS\n\nBALL CORPORATION\n\nUNAUDITED CONDENSED CONSOLIDATED STATEMENTS OF EARNINGS\n[Chunk Index 1] ######Three Months Ended September 30,##########Nine Months Ended September 30,####\n($ in millions, except per share amounts)####2023######2022####2023######2022\nNet sales##$##3,571####$##3,951##$##10,626####$##11,801\nCosts and expenses####################\nCost of sales (excluding depreciation and amortization)####(2,894)######(3,275)####(8,655)######(9,736)\nDepreciation and amortization####(173)######(157)####(509)######(510)\nSelling, general and administrative####(132)######(159)####(428)######(506)\nBusiness consolidation and other activities####

In [ ]:
df_chunk_eval["messages"][0][0]['content']

"Identify the 10 most relevant text chunks for answering this question, then rank them in order of relevance (best first).\nQuestion: What guidance was offered on Ball Corporation’s aluminum can production capacity targets?\nText chunks:\n[Chunk Index 0] PART I. FINANCIAL INFORMATION\n\nItem 1. FINANCIAL STATEMENTS\n\nBALL CORPORATION\n\nUNAUDITED CONDENSED CONSOLIDATED STATEMENTS OF EARNINGS\n[Chunk Index 1] ######Three Months Ended September 30,##########Nine Months Ended September 30,####\n($ in millions, except per share amounts)####2023######2022####2023######2022\nNet sales##$##3,571####$##3,951##$##10,626####$##11,801\nCosts and expenses####################\nCost of sales (excluding depreciation and amortization)####(2,894)######(3,275)####(8,655)######(9,736)\nDepreciation and amortization####(173)######(157)####(509)######(510)\nSelling, general and administrative####(132)######(159)####(428)######(506)\nBusiness consolidation and other activities####(47)######163####(61)#####

In [ ]:
"""calculate max len of strings in df_chunk_eval["messages"][x][0]['content'] where x is row"""
max_len = 0
for x in range(len(df_chunk_eval)):
  if len(df_chunk_eval["messages"][x][0]['content']) > max_len:
    max_len = len(df_chunk_eval["messages"][x][0]['content'])
max_len

1050493

In [ ]:
i = df_chunk_eval["messages"][0][0]['content']
ranking_chunk = deepseek_model_chunks(system_prompt_chunk, i)
print(ranking_chunk)

<coroutine object deepseek_model_chunks at 0x7d0b11896d40>


In [ ]:
# convert string of a list to list
import ast
ast.literal_eval(ranking_chunk.choices[0].message.content)

[92, 93, 94, 95, 96, 97, 98, 99, 100, 101]

In [ ]:
print(ranking_chunk.choices[0].message.content)

[92, 93, 94, 95, 96, 97, 98, 99, 100, 101]


## Get Chunks ranking

In [ ]:
def create_chunk_prompt_top_k(question: str, chunks: List[str], chunk_indices: List[int], k: int = 10) -> str:
    """Ask model to select and rank only top-k most relevant chunks"""
    # Use k if chunks length > 10, else use chunks length
    actual_k = k if len(chunks) > 10 else len(chunks)

    prompt = f"""Identify the {actual_k} most relevant text chunks for answering this question, then rank them in order of relevance (best first).
Question: {question}
Text chunks:
"""
    for i, (chunk, orig_idx) in enumerate(zip(chunks, chunk_indices)):
        prompt += f"[Chunk Index {orig_idx}] {chunk}\n"
    prompt += f"""
Task: Select and rank the {actual_k} most relevant chunks among the given text chunks.
- Put the BEST chunk first
- Put the 2nd best chunk second
- Continue until you have ranked your top {actual_k} chunks
Response Format: [1st_most_relevant_index, 2nd_most_relevant_index, ..., {actual_k}th_most_relevant_index]"""

    return prompt

In [ ]:
async def process_chunk_ranking_two_stage(system_prompt, messages):
  """Process chunk ranking with multi-stage approach for high token count cases"""
  try:
      # Check if this is a high token case by examining the message content
      encoding = tiktoken.get_encoding("cl100k_base")
      # content = messages[0].get('content', '')
      content = messages
      token_count = len(encoding.encode(content))

      if token_count > 120000:

          # Extract question and chunks from the message content
          # Find question
          question_start = content.find('Question:')
          question_end = content.find('\n', question_start)
          if question_start != -1 and question_end != -1:
              question = content[question_start + len('Question:'):question_end].strip()
          else:
              question = None

          # Find chunks using regex-like pattern matching
          chunks = []
          chunk_indices = []

          import re
          # Pattern to match [Chunk Index N] followed by content until next [Chunk Index] or Task:
          chunk_pattern = r'\[Chunk Index (\d+)\]\s*([\s\S]*?)(?=\[Chunk Index|Task:|$)'
          matches = re.findall(chunk_pattern, content)

          for i, match in enumerate(matches):
              orig_idx = int(match[0])
              chunk_content = match[1].strip()

              # Clean up chunk content - remove any task instructions that might be caught
              if 'Task:' in chunk_content:
                  chunk_content = chunk_content.split('Task:')[0].strip()

              if chunk_content:
                  chunks.append(chunk_content)
                  chunk_indices.append(orig_idx)

          if not question or not chunks:
              print("⚠️ Could not parse question and chunks, falling back to normal processing")
              final_response = await deepseek_model_chunks(system_prompt, messages)
              # response = await get_model_response(messages, semaphore=semaphore)
              # predicted_ranking = extract_ranking_from_response(response, 10)
          else:
              # Split chunks into three parts
              third_point_1 = len(chunks) // 3
              third_point_2 = (len(chunks) * 2) // 3

              # First third
              first_third_chunks = chunks[:third_point_1]
              first_third_indices = chunk_indices[:third_point_1]
              first_prompt = create_chunk_prompt_top_k(question, first_third_chunks, first_third_indices, k=10)
              # first_messages = [{"role": "user", "content": first_prompt}]
              # first_response = await get_model_response(first_messages, semaphore=semaphore)
              first_top_3 = await deepseek_model_chunks(system_prompt, first_prompt)
              first_top_3 = ast.literal_eval(first_top_3.choices[0].message.content)
              # first_top_3 = extract_ranking_from_response(first_response, 10)

              # Second third
              second_third_chunks = chunks[third_point_1:third_point_2]
              second_third_indices = chunk_indices[third_point_1:third_point_2]
              second_prompt = create_chunk_prompt_top_k(question, second_third_chunks, second_third_indices, k=10)
              # second_messages = [{"role": "user", "content": second_prompt}]
              # second_response = await get_model_response(second_messages, semaphore=semaphore)
              second_top_3 = await deepseek_model_chunks(system_prompt, second_prompt)
              second_top_3 = ast.literal_eval(second_top_3.choices[0].message.content)
              # second_top_3 = extract_ranking_from_response(second_response, 10)

              # Third third
              third_third_chunks = chunks[third_point_2:]
              third_third_indices = chunk_indices[third_point_2:]
              third_prompt = create_chunk_prompt_top_k(question, third_third_chunks, third_third_indices, k=10)
              # third_messages = [{"role": "user", "content": third_prompt}]
              # third_response = await get_model_response(third_messages, semaphore=semaphore)
              third_top_4 = await deepseek_model_chunks(system_prompt, third_prompt)
              third_top_4 = ast.literal_eval(third_top_4.choices[0].message.content)
              # third_top_4 = extract_ranking_from_response(third_response, 10)

              # Combine top results from each third
              combined_indices = first_top_3 + second_top_3 + third_top_4
              combined_chunks = []

              # Get chunks for the combined indices while preserving original indices
              for idx in combined_indices:
                  if idx in chunk_indices:
                      chunk_pos = chunk_indices.index(idx)
                      combined_chunks.append(chunks[chunk_pos])

              final_prompt = create_chunk_prompt_top_k(question, combined_chunks, combined_indices, k=10)
              # final_messages = [{"role": "user", "content": final_prompt}]
              final_response = await deepseek_model_chunks(system_prompt, final_prompt) #, semaphore=semaphore)
              # predicted_ranking = extract_ranking_from_response(final_response, 10)

      else:
          # Normal single-stage processing
          final_response = await deepseek_model_chunks(system_prompt, messages) #, semaphore=semaphore)
          # predicted_ranking = extract_ranking_from_response(response, 10)

      return final_response
  except Exception as e:
      traceback.print_exc()
      print(f"❌ Error processing chunk ranking item: {e}")
      return []

In [ ]:
# results = pd.DataFrame(columns=['sample_id', 'target_index'])

In [ ]:
rankings_chunk = []
for i in range(115, len(df_chunk_eval)):
  prompt = df_chunk_eval["messages"][i][0]['content']
  ranking_chunk = await process_chunk_ranking_two_stage(system_prompt_chunk, prompt)
  current_rank_chunks = ast.literal_eval(ranking_chunk.choices[0].message.content)
  rankings_chunk.append(current_rank_chunks)

  if len(current_rank_chunks)<=5:
    current_rank_chunks = current_rank_chunks + [0]*(5-len(current_rank_chunks))

  print(current_rank_chunks)
  for j in current_rank_chunks[:5]:
    current_id = df_chunk_eval['_id'][i]
    chunk_rank = {'sample_id': current_id, 'target_index': j}
    # add chunk rank to df
    results = pd.concat([results, pd.DataFrame([chunk_rank])], ignore_index=True)

df_chunk_eval["rankings"] = rankings_chunk

[32, 33, 34, 35, 38, 39, 40, 89, 90, 114]
[7, 6, 9, 1, 16, 46, 47, 44, 45, 240]
[10, 2, 0, 9, 4, 11, 6, 8, 3, 7]
[5, 4, 6, 7, 24, 25, 26, 27, 28, 29]
[1, 0, 2, 27, 28, 29, 30, 31, 32, 33]
[4, 7, 9, 10, 11, 28, 30, 1, 8, 12]
[5, 7, 57, 67, 77, 91, 101, 107, 19, 35]
[67, 65, 66, 68, 69, 70, 71, 72, 64, 73]
[2, 3, 9, 8, 30, 31, 32, 33, 34, 35]
[75, 15, 142, 137, 138, 139, 140, 141, 0, 1]
[14, 15, 27, 33, 57, 59, 5, 7, 51, 13]
[194, 195, 196, 197, 198, 199, 115, 116, 117, 118]
[1, 107, 108, 109, 110, 111, 112, 113, 114, 115]
[60, 61, 128, 129, 7, 62, 63, 64, 65, 66]
[52, 53, 54, 55, 56, 57, 58, 59, 60, 61]
[101, 126, 55, 9, 144, 145, 146, 138, 139, 142]
[38, 42, 487, 486, 485, 39, 40, 43, 44, 41]
[1, 6, 4, 10, 5, 0, 2, 3, 7, 9]
[6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
[5, 7, 3, 4, 6, 122, 123, 124, 125, 0]
[15, 16, 17, 18, 1, 2, 4, 5, 6, 7]
[97, 96, 110, 111, 112, 101, 108, 185, 186, 98]
[43, 42, 41, 44, 45, 47, 48, 49, 50, 51]
[6, 7, 4, 1, 2, 3, 5, 0, 8, 9]
[0, 5, 15, 1, 37, 38, 45, 46, 48, 4

ValueError: Length of values (85) does not match length of index (200)

In [ ]:
# Convert string to list of lists
ranking_lists_all = [ast.literal_eval(line) for line in ranking_list_temp.strip().splitlines() if line.strip()]

print(ranking_lists_all)

[[96, 98, 99, 100, 102, 103, 104, 106, 107, 108], [8, 9, 137, 138, 139, 140, 141, 142, 143, 144], [7, 5, 12, 45, 33, 19, 61, 67, 13, 15], [2, 3, 4, 5, 6, 7, 8, 181, 182, 1], [21, 22, 20, 6, 5, 1, 2, 3, 4, 7], [5, 7, 57, 113, 57, 5, 7, 113, 5, 7], [11, 17, 23, 29, 35, 43, 49, 55, 61, 67], [4, 1, 3, 5, 9, 6, 2, 0, 7, 8], [150, 149, 41, 230, 233, 231, 232, 234, 235, 236], [29, 30, 31, 32, 33, 34, 35, 36, 37, 38], [77, 76, 90, 89, 91, 56, 57, 78, 79, 80], [3, 2, 1, 4, 5, 6, 7, 8, 9, 10], [9, 66, 68, 16, 6, 8, 5, 2, 1, 15], [4, 20, 19, 24, 25, 3, 0, 1, 2, 5], [31, 30, 29, 28, 32, 33, 34, 35, 36, 37], [43, 44, 45, 46, 47, 48, 49, 50, 51, 52], [7, 5, 89, 87, 91, 93, 101, 103, 105, 107], [5, 7, 11, 57, 23, 27, 61, 67, 89, 101], [1, 3, 11, 9, 6, 12, 7, 4, 13, 2], [3, 1, 0, 2, 4], [17, 6, 5, 16, 15, 18, 19, 20, 21, 22], [10, 13, 16, 9, 12, 15, 14, 11, 18, 17], [253, 647, 4, 42, 64, 75, 237, 238, 252, 489], [3, 0, 23, 24, 25, 26, 27, 4, 1, 2], [16, 14, 13, 15, 11, 12, 1, 2, 10, 9], [86, 29, 30, 9

In [ ]:
len(ranking_lists_all)

200

In [ ]:
results = pd.DataFrame(columns=['sample_id', 'target_index'])


for index, i in enumerate(ranking_lists_all):
  if len(i)<=5:
      i = i + [0]*(5-len(i))

  # print(i)
  for j in i[:5]:
    current_id = df_chunk_eval['_id'][index]
    chunk_rank = {'sample_id': current_id, 'target_index': j}
    # add chunk rank to df
    results = pd.concat([results, pd.DataFrame([chunk_rank])], ignore_index=True)

In [ ]:
results

,sample_id,target_index
0,chunk_q613509,96
1,chunk_q613509,98
2,chunk_q613509,99
3,chunk_q613509,100
4,chunk_q613509,102
...,...,...
995,chunk_qa1d2bd,17
996,chunk_qa1d2bd,0
997,chunk_qa1d2bd,23
998,chunk_qa1d2bd,25


In [ ]:
# save results with chunks
results.to_csv('results_chunks.csv', index=False)

In [ ]:
# join results.csv and results_chunks.csv
df_results = pd.read_csv('results.csv')

# change ranking col to target_index
df_results = df_results.rename(columns={'rankings': 'target_index'}, inplace=False)
df_results_chunks = pd.read_csv('results_chunks.csv')

# join
df_results_all = pd.concat([df_results, df_results_chunks], ignore_index=True)

In [ ]:
df_results_all.to_csv('results_all.csv', index=False)